# Notebook 03 — QLoRA: quantize the base, then LoRA on top

**Workshop:** Agentic AI for Scientists — Week 4 (Post-Training & Deployment)
**Block:** quantization + the three-way comparison (20 min)
**Goal:** Combine **4-bit quantization** of the frozen base model with LoRA adapters. This is the technique behind "fine-tune a 7B/13B model on a free Colab GPU." Then we put **full SFT vs LoRA vs QLoRA** side by side.

**What quantization is:** store each weight in **4 bits** instead of 16. A 0.6B model's base shrinks ~4x in memory. QLoRA uses two tricks from the original paper (Dettmers et al., 2023):

- **NF4 (NormalFloat-4)** — a 4-bit datatype shaped to the bell-curve distribution of neural-net weights, so it loses less information than naive int4.
- **Double quantization** — quantize the quantization constants too. Small extra saving, free accuracy.

The adapters stay in 16-bit and do the learning; the quantized base is only ever read (de-quantized on the fly during the forward/backward pass). That on-the-fly de-quant is why QLoRA trades a little **speed** for a lot of **memory**.

## 0. Setup & GPU check

In [ ]:
%pip install -q unsloth
%pip install -q --no-deps "trl<0.24" peft accelerate bitsandbytes datasets

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU, then re-run."
)
gpu = torch.cuda.get_device_properties(0)
print(f"GPU            : {gpu.name}")
print(f"Total VRAM     : {gpu.total_memory / 1024**3:.1f} GB")
print(f"Compute Cap.   : {gpu.major}.{gpu.minor}  "
      f"({'Ampere+ (Flash-Attention 2 path)' if gpu.major >= 8 else 'pre-Ampere (T4): works, slower attention path'})")
print(f"bf16 supported : {torch.cuda.is_bf16_supported()}")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

---
## 1. Load the base model in 4-bit, then add LoRA

The single line that turns LoRA into QLoRA: `load_in_4bit=True`. Unsloth configures NF4 + double-quant + the compute dtype automatically — do **not** pass the `bnb_*` kwargs by hand (it errors).

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-0.6B",
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    load_in_4bit=True,          # <-- QLoRA: 4-bit NF4 base (Unsloth sets double-quant + nf4)
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_train/1e6:.3f}M  ({100*n_train/n_total:.2f}%)  on a 4-bit base")

## 2. Dataset (identical) + train

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset, Dataset

SYSTEM_PROMPT = (
    "You are a clinical assistant. Read the medical question and respond with a JSON object "
    "containing: disease, patient_info, symptoms (list), treatment (list), answer_summary."
)
def load_prepared(split):
    p = Path(f"data/medquad_{split}.jsonl")
    if p.exists():
        return Dataset.from_list([json.loads(l) for l in p.read_text().splitlines() if l.strip()])
    return None
train_ds, eval_ds = load_prepared("train"), load_prepared("validation")
if train_ds is None:
    raw = load_dataset("lavita/MedQuAD", split="train").filter(lambda r: r.get("answer"))
    raw = raw.shuffle(seed=3407).select(range(200))
    rows = [{"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": r["question"]},
        {"role": "assistant", "content": r["answer"][:800]}]} for r in raw]
    train_ds, eval_ds = Dataset.from_list(rows[:160]), Dataset.from_list(rows[160:])
def to_text(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)}
train_ds, eval_ds = train_ds.map(to_text), eval_ds.map(to_text)
print(f"train: {len(train_ds)} | eval: {len(eval_ds)}")

In [ ]:
from trl import SFTTrainer, SFTConfig
import torch, time

cfg = SFTConfig(
    output_dir="outputs_qlora",
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    warmup_steps=5, max_steps=60, learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit", weight_decay=0.01,
    logging_steps=2, eval_strategy="steps", eval_steps=30, save_strategy="no",
    seed=3407, report_to="none", max_seq_length=MAX_SEQ_LEN, dataset_text_field="text",
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=eval_ds, args=cfg)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
stats = trainer.train()
elapsed = time.time() - t0
peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"\nFinal train loss : {stats.training_loss:.4f}")
print(f"Peak VRAM        : {peak:.2f} GB  (lowest of the three — that's the point)")
print(f"Wall time        : {elapsed:.0f}s  (a touch slower per step: de-quant overhead)")

Path("data").mkdir(exist_ok=True)
Path("data/run_qlora.json").write_text(json.dumps(
    {"method": "qlora", "trainable_pct": round(100*n_train/n_total, 2), "peak_vram_gb": round(peak, 2),
     "final_loss": round(float(stats.training_loss), 4), "steps": stats.global_step,
     "wall_s": round(elapsed)}, indent=2))

---
## 3. Merge to a standalone 16-bit model (for easy deployment)

QLoRA trains adapters on a 4-bit base, but to deploy with *any* framework it's convenient to **merge** the adapters into the base and export a normal 16-bit model. (You can also keep base+adapter separate and serve them that way — NB04 shows both.)

In [ ]:
# Merge LoRA into the base and save a standalone fp16 model.
model.save_pretrained_merged("qwen3-medquad-qlora-merged", tokenizer, save_method="merged_16bit")
import os
print("Merged 16-bit model saved to ./qwen3-medquad-qlora-merged")

# Optional Hub push of the merged model:
# model.push_to_hub_merged("YOUR_USERNAME/qwen3-medquad-qlora", tokenizer, save_method="merged_16bit", token=os.environ["HF_TOKEN"])

---
## 4. The three-way comparison

This is the slide that makes the whole week land. We read back the `data/run_*.json` files each notebook wrote and tabulate them. (If you only ran this notebook, the missing rows are filled with the reference numbers from a clean A-tier run so the shape is still visible — re-run NB01/NB02 to populate your own.)

In [ ]:
import json
from pathlib import Path

# Reference numbers from a representative run (Qwen3-0.6B, 60 steps, T4) — used as
# fallback ONLY when you haven't run that notebook in this session.
REF = {
    "full_sft": {"method": "full_sft", "trainable_pct": 100.0, "peak_vram_gb": 5.9, "final_loss": 1.21, "steps": 60},
    "lora":     {"method": "lora",     "trainable_pct": 1.95,  "peak_vram_gb": 3.4, "final_loss": 1.28, "steps": 60},
    "qlora":    {"method": "qlora",    "trainable_pct": 1.95,  "peak_vram_gb": 2.3, "final_loss": 1.31, "steps": 60},
}

def load_run(name):
    p = Path(f"data/run_{name}.json")
    if p.exists():
        d = json.loads(p.read_text()); d["_source"] = "your run"; return d
    d = dict(REF[name]); d["_source"] = "reference"; return d

rows = [load_run(n) for n in ["full_sft", "lora", "qlora"]]

hdr = f"{'method':10s} {'trainable%':>11s} {'peak VRAM':>10s} {'final loss':>11s} {'source':>10s}"
print(hdr); print("-" * len(hdr))
for r in rows:
    print(f"{r['method']:10s} {r['trainable_pct']:>10.2f}% {r['peak_vram_gb']:>8.2f}GB "
          f"{r['final_loss']:>11.3f} {r['_source']:>10s}")
print()
print("Takeaways:")
print("  - LoRA  : ~1-2% trainable, big VRAM cut, near-full-FT quality.")
print("  - QLoRA : lowest VRAM (4-bit base), slightly slower, tiny quality cost.")
print("  - Full  : best quality ceiling, heaviest cost. Use when you can afford it.")

## What you learned

Quantization (NF4 + double-quant) shrinks the frozen base ~4x, so QLoRA gives you the smallest VRAM footprint of the three — the reason a single consumer GPU can now fine-tune models that used to need a server.

You now have three trained models and a comparison table. But "lower loss" is not "good clinical answers." **Whether the fine-tune actually worked is a measurement question — and that is the entire subject of Week 5.**

**Next — `04_deploy_inference.ipynb`:** load a tuned model and look at the real ways to *serve* it (local, endpoint, on-device) and what each costs.